In [ ]:
import pandas as pd
import numpy as np

import asf_lifetime_cost_model.getters.data_getters as data_getters

from typing import Optional

In [2]:
# Helper function to extract heat demand from table for given archetype


def _get_property_heat_demand(archetype_number: int) -> float:

    # Get property heat demand data
    heat_demand_data = data_getters.get_property_heat_demand()

    # Extract annual heat demand of selected property archetype
    property_heat_demand = heat_demand_data.loc[
        heat_demand_data["archetype_number"] == archetype_number, "heat_demand"
    ].iloc[
        0
    ]  # kWh/year

    return property_heat_demand

In [3]:
# Helper function to extract wholesale price projection from table for given year, fuel type and projection scenario


def _get_wholesale_price_projection_series(fuel_type: str, projection_scenario: str):

    # Get DESNZ gas and electricity wholesale price projections data
    wholesale_prices_data = data_getters.get_desnz_wholesale_price_projections(
        projection_scenario_names=[projection_scenario]
    )

    if projection_scenario == "reference":
        # Modify data so that wholesale prices are held constant at 2040 levels from 2041 to 2050 for reference scenario
        for year in range(2041, 2051):
            wholesale_prices_data.loc[
                wholesale_prices_data["projection scenario"] == "reference", year
            ] = wholesale_prices_data.loc[
                wholesale_prices_data["projection scenario"] == "reference", 2040
            ]
    elif (projection_scenario == "low fossil fuel prices") | (
        projection_scenario == "high fossil fuel prices"
    ):
        # Modify dataset so that wholesale prices are held constant at 2031 levels from 2032 to 2050 for low and high fossil fuel prices scenarios
        for year in range(2032, 2051):
            wholesale_prices_data.loc[
                wholesale_prices_data["projection scenario"]
                == "low fossil fuel prices",
                year,
            ] = wholesale_prices_data.loc[
                wholesale_prices_data["projection scenario"]
                == "low fossil fuel prices",
                2031,
            ]
            wholesale_prices_data.loc[
                wholesale_prices_data["projection scenario"]
                == "high fossil fuel prices",
                year,
            ] = wholesale_prices_data.loc[
                wholesale_prices_data["projection scenario"]
                == "high fossil fuel prices",
                2031,
            ]
    else:
        raise ValueError("Invalid projection scenario name provided.")

    wholesale_prices_series = wholesale_prices_data[
        wholesale_prices_data["fuel"].str.contains(fuel_type, case=False)
    ]

    return wholesale_prices_series

In [6]:
# Helper function


def _create_energy_cost_time_series(
    tariff_time_series: dict,
    fuel_type: str,
    wholesale_prices_series: pd.DataFrame,
) -> tuple[dict[int, float], dict[int, float]]:

    unit_cost_time_series = {}
    standing_charge_time_series = {}

    for year, tariff in tariff_time_series.items():

        # Cost component to broader category mapping
        tariff_cost_components_category_map = {
            "nil": {
                tariff.df_nil: "wholesale",
                tariff.cm_nil: "wholesale",
                tariff.aa_nil: "other",
                tariff.pc_nil: "policy",
                tariff.nc_nil: "other",
                tariff.oc_nil: "other",
                tariff.smncc_nil: "other",
                tariff.ic_nil: "other",
                tariff.paac_nil: "other",
                tariff.pap_nil: "other",
                tariff.co_nil: "other",
                tariff.drc_nil: "other",
                tariff.ebit_nil: "other",
                tariff.hap_nil: "other",
                tariff.levelisation_nil: "other",
            },
            "variable": {
                tariff.df: "wholesale",
                tariff.cm: "wholesale",
                tariff.aa: "other",
                tariff.pc: "policy",
                tariff.nc: "other",
                tariff.oc: "other",
                tariff.smncc: "other",
                tariff.ic: "other",
                tariff.paac: "other",
                tariff.pap: "other",
                tariff.co: "other",
                tariff.drc: "other",
                tariff.ebit: "other",
                tariff.hap: "other",
                tariff.levelisation: "other",
            },
        }

        # Create dictionary where tariff cost components are aggregated by category
        categories = ["wholesale", "policy", "other"]
        types = ["nil", "variable"]
        tariff_cost_categories = {
            f"{cat} ({type})": np.nansum(
                [
                    component
                    for component, category in tariff_cost_components_category_map.get(
                        type
                    ).items()
                    if category == cat
                ]
            )
            for type in types
            for cat in categories
        }

        # Look up wholesale price projection for year
        # But do not replace if year is year of current price cap period
        if year != tariff.price_cap_period.left.year:

            if fuel_type == "gas":
                # Convert p/therm to p/kWh
                therm_to_kwh_conversion = 29.31

                wholesale_projection = (
                    wholesale_prices_series.loc[:, year].iloc[0]
                ) / therm_to_kwh_conversion  # p/kWh

            elif fuel_type == "electricity":
                wholesale_projection = wholesale_prices_series.loc[:, year].iloc[
                    0
                ]  # p/kWh

            else:
                raise ValueError("Invalid fuel type.")

            # Replace wholesale component with projection
            tariff_cost_categories["wholesale (variable)"] = (
                wholesale_projection * 10
            )  # convert p/kWh to £/MWh

        else:
            pass

        # Compute total unit cost and standing charge for the year
        unit_cost = np.nansum(
            [
                value
                for category, value in tariff_cost_categories.items()
                if "(variable)" in category
            ]
        )
        standing_charge = np.nansum(
            [
                value
                for category, value in tariff_cost_categories.items()
                if "(nil)" in category
            ]
        )

        unit_cost_time_series[year] = unit_cost
        standing_charge_time_series[year] = standing_charge

    return unit_cost_time_series, standing_charge_time_series

In [7]:
# Main function
def compute_running_costs(
    purchase_year: int,
    life_span: int,
    heating_system_efficiency: float,
    fuel_type: str,
    archetype_number: int,
    wholesale_price_projection_scenario: str,
    include_standing_charge: bool,
    levy_rebalancing: bool,
    levies_to_rebalance: Optional[list[str]] = None,
    levy_rebalancing_weights: Optional[dict[str, float]] = None,
) -> dict[int, float]:
    "Get heat and energy demand"

    # Get annual heat demand of property archetype
    property_heat_demand = _get_property_heat_demand(archetype_number)  # kWh/year

    # Calculate annual energy demand of heating system
    energy_demand = property_heat_demand / heating_system_efficiency

    "Create energy tariff for each year of operation"

    # Instantiate Tariff objects for latest price cap
    gas_tariff, electricity_tariff = data_getters.get_current_energy_price_cap_tariffs()

    # Apply levy rebalancing scenario to latest policy costs
    if levy_rebalancing:
        rebalanced_levies = data_getters.get_rebalanced_levies(
            levies_to_rebalance=levies_to_rebalance,
            electricity_weight=levy_rebalancing_weights["electricity_weight"],
            gas_weight=levy_rebalancing_weights["gas_weight"],
            tax_weight=levy_rebalancing_weights["tax_weight"],
            variable_electricity_weight=levy_rebalancing_weights[
                "variable_electricity_weight"
            ],
            fixed_electricity_weight=levy_rebalancing_weights[
                "fixed_electricity_weight"
            ],
            variable_gas_weight=levy_rebalancing_weights["variable_gas_weight"],
            fixed_gas_weight=levy_rebalancing_weights["fixed_gas_weight"],
            price_cap_period="LATEST",
        )
        gas_tariff = gas_tariff.update_policy_costs(rebalanced_levies)
        electricity_tariff = electricity_tariff.update_policy_costs(rebalanced_levies)
    else:
        pass

    # Create list of operating years
    years_of_operation = list(range(purchase_year, purchase_year + life_span))

    # Create a dictionary of gas and electricity tariffs for each year of operation
    # Note we are using the same tariff for each year in the future
    gas_tariff_time_series = {}
    electricity_tariff_time_series = {}
    for year in years_of_operation:
        gas_tariff_time_series[year] = gas_tariff
        electricity_tariff_time_series[year] = electricity_tariff

    "Get wholesale price projections"

    # Get wholesale price projection time series for fuel and scenario of interest
    wholesale_prices_series_df = _get_wholesale_price_projection_series(
        fuel_type=fuel_type, projection_scenario=wholesale_price_projection_scenario
    )

    "Aggregate tariff cost components to replace wholesale price with projection"
    # Wholesale costs include DF and CM tariff cost components
    # We need to aggregate the tariff cost components by broader categories to be able to
    # replace the wholesale costs with the projection for that year

    # Gas costs
    gas_unit_cost_time_series, gas_standing_charge_time_series = (
        _create_energy_cost_time_series(
            tariff_time_series=gas_tariff_time_series,
            fuel_type="gas",
            wholesale_prices_series=wholesale_prices_series_df,
        )
    )

    # Electricity costs
    electricity_unit_cost_time_series, electricity_standing_charge_time_series = (
        _create_energy_cost_time_series(
            tariff_time_series=electricity_tariff_time_series,
            fuel_type="electricity",
            wholesale_prices_series=wholesale_prices_series_df,
        )
    )

    "Compute total running cost in each year of operation"
    running_costs_time_series = {}
    for year in years_of_operation:
        if fuel_type == "gas":
            running_costs_time_series[year] = gas_unit_cost_time_series[year] * (
                energy_demand / 1_000
            )
            if include_standing_charge:
                running_costs_time_series[year] += gas_standing_charge_time_series[year]
        elif fuel_type == "electricity":
            running_costs_time_series[year] = electricity_unit_cost_time_series[
                year
            ] * (energy_demand / 1_000)
            if include_standing_charge:
                running_costs_time_series[
                    year
                ] += electricity_standing_charge_time_series[year]

    return running_costs_time_series

In [8]:
boiler_running_costs_time_series = compute_running_costs(
    purchase_year=2025,
    life_span=15,
    heating_system_efficiency=0.85,
    fuel_type="gas",
    archetype_number=1,
    wholesale_price_projection_scenario="reference",
    include_standing_charge=True,
    levy_rebalancing=False,
)

2025-08-11 17:17:26,929 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [9]:
sum(boiler_running_costs_time_series.values())

np.float64(7012.372382650804)

In [10]:
ashp_running_costs_time_series = compute_running_costs(
    purchase_year=2025,
    life_span=15,
    heating_system_efficiency=3.0,
    fuel_type="electricity",
    archetype_number=1,
    wholesale_price_projection_scenario="reference",
    include_standing_charge=False,
    levy_rebalancing=False,
)

In [11]:
sum(ashp_running_costs_time_series.values())

np.float64(5908.488099996602)

---

### Development

Inputs
- archetype number -> lookup heat demand
- fossil fuel prices scenario -> DESNZ scenario -> replace wholesale cost components in tariff
- levy rebalancing scenario -> create rebalanced levycollection -> replace policy cost component in tariff
- year of purchase -> start and end years of operation

Calculation
- Use ASHP efficiency to determine electricity demand for heating and use boiler efficiency to determine gas demand for heating
- Multiply electricity/gas demand for heating by unit cost of electricity/gas
- Add standing charge for gas boiler

In [ ]:
# Universal parameters from config
life_span = 15
ashp_efficiency = 3.0
boiler_efficiency = 0.85

**Heat demand and energy demand**

In [ ]:
# get property heat demand data
heat_demand_data = data_getters.get_property_heat_demand()

In [ ]:
heat_demand_data

In [ ]:
# input archetype
archetype_number = 1

In [ ]:
heat_demand = heat_demand_data.loc[
    heat_demand_data["archetype_number"] == archetype_number, "heat_demand"
].iloc[
    0
]  # kWh/year

In [ ]:
ashp_electricity_demand = heat_demand / ashp_efficiency
ashp_electricity_demand  # kWh/year

In [ ]:
boiler_gas_demand = heat_demand / boiler_efficiency
boiler_gas_demand  # kWh/year

**Cost of energy each year of operation**

In [ ]:
# Instantiate Tariff objects for current price cap
gas_tariff, electricity_tariff = data_getters.get_current_energy_price_cap_tariffs()

Replace policy costs with levy rebalancing scenario

In [ ]:
# input: rebalancing scenario name
# rebalancing_scenario_name = "rebalance ro and fit to gas"

# rebalanced_levies = data_getters.get_rebalanced_levies(rebalancing_scenario_name)
# gas_tariff = gas_tariff.update_policy_costs(rebalanced_levies)
# electricity_tariff = electricity_tariff.update_policy_costs(rebalanced_levies)

Create time series

In [ ]:
# input: year of purchase
purchase_year = 2025

# create list of range of operating years
years_of_operation = list(range(purchase_year, purchase_year + life_span))

In [ ]:
# Create a dictionary for gas tariffs and electricity tariffs for each year of operation
gas_tariff_each_year = {}
electricity_tariff_each_year = {}
for year in years_of_operation:
    gas_tariff_each_year[year] = gas_tariff
    electricity_tariff_each_year[year] = electricity_tariff

Replacing the wholesale cost components of tariffs with DESNZ projections

In [ ]:
# desnz wholesale price projections
wholesale_prices_data = data_getters.get_desnz_wholesale_price_projections(
    projection_scenario_names=[
        "reference",
        "low fossil fuel prices",
        "high fossil fuel prices",
    ]
)

In [ ]:
# modify dataset so that wholesale prices are held constant at 2040 levels from 2041 to 2050 for reference scenario
for year in range(2041, 2051):
    wholesale_prices_data.loc[
        wholesale_prices_data["projection scenario"] == "reference", year
    ] = wholesale_prices_data.loc[
        wholesale_prices_data["projection scenario"] == "reference", 2040
    ]

In [ ]:
# modify dataset so that wholesale prices are held constant at 2031 levels from 2032 to 2050 for low and high fossil fuel prices scenarios
for year in range(2032, 2051):
    wholesale_prices_data.loc[
        wholesale_prices_data["projection scenario"] == "low fossil fuel prices", year
    ] = wholesale_prices_data.loc[
        wholesale_prices_data["projection scenario"] == "low fossil fuel prices", 2031
    ]

for year in range(2032, 2051):
    wholesale_prices_data.loc[
        wholesale_prices_data["projection scenario"] == "high fossil fuel prices", year
    ] = wholesale_prices_data.loc[
        wholesale_prices_data["projection scenario"] == "high fossil fuel prices", 2031
    ]

In [ ]:
# input: wholesale price projection scenario
wholesale_price_projection_scenario = "reference"

Creating time series of energy costs

In [ ]:
# creating time series of gas costs
gas_unit_costs_each_year = {}
gas_standing_charges_each_year = {}
for year, gas_tariff in gas_tariff_each_year.items():

    # Cost component category mapping
    gas_tariff_cost_components_category_map = {
        "nil": {
            gas_tariff.df_nil: "wholesale",
            gas_tariff.cm_nil: "wholesale",
            gas_tariff.aa_nil: "other",
            gas_tariff.pc_nil: "policy",
            gas_tariff.nc_nil: "other",
            gas_tariff.oc_nil: "other",
            gas_tariff.smncc_nil: "other",
            gas_tariff.ic_nil: "other",
            gas_tariff.paac_nil: "other",
            gas_tariff.pap_nil: "other",
            gas_tariff.co_nil: "other",
            gas_tariff.drc_nil: "other",
            gas_tariff.ebit_nil: "other",
            gas_tariff.hap_nil: "other",
            gas_tariff.levelisation_nil: "other",
        },
        "variable": {
            gas_tariff.df: "wholesale",
            gas_tariff.cm: "wholesale",
            gas_tariff.aa: "other",
            gas_tariff.pc: "policy",
            gas_tariff.nc: "other",
            gas_tariff.oc: "other",
            gas_tariff.smncc: "other",
            gas_tariff.ic: "other",
            gas_tariff.paac: "other",
            gas_tariff.pap: "other",
            gas_tariff.co: "other",
            gas_tariff.drc: "other",
            gas_tariff.ebit: "other",
            gas_tariff.hap: "other",
            gas_tariff.levelisation: "other",
        },
    }

    # Aggregate costs by category
    categories = ["wholesale", "policy", "other"]
    types = ["nil", "variable"]
    gas_tariff_cost_categories = {
        f"{cat} cost ({type})" if cat == "policy" else f"{cat} ({type})": np.nansum(
            [
                component
                for component, category in gas_tariff_cost_components_category_map.get(
                    type, {}
                ).items()
                if category == cat
            ]
        )
        for type in types
        for cat in categories
    }

    # p/therm to p/kWh
    therm_to_kwh_conversion = 29.31

    # look up wholesale price projection for year of interest
    # do not replace if year of interest is year of price cap period
    if year != gas_tariff.price_cap_period.left.year:
        gas_wholesale_projection = (
            wholesale_prices_data.loc[
                (
                    wholesale_prices_data["projection scenario"]
                    == wholesale_price_projection_scenario
                )
                & (wholesale_prices_data["fuel"].str.contains("gas", case=False)),
                year,
            ].iloc[0]
            / therm_to_kwh_conversion
        )  # p/kWh

        # replace wholesale components with DESNZ projections
        gas_tariff_cost_categories["wholesale (variable)"] = (
            gas_wholesale_projection * 10
        )  # convert p/kWh to £/MWh
    else:
        pass

    # extract total unit costs and standing charges
    gas_unit_cost = np.nansum(
        [
            value
            for category, value in gas_tariff_cost_categories.items()
            if "(variable)" in category
        ]
    )
    gas_standing_charge = np.nansum(
        [
            value
            for category, value in gas_tariff_cost_categories.items()
            if "(nil)" in category
        ]
    )

    gas_unit_costs_each_year[year] = gas_unit_cost
    gas_standing_charges_each_year[year] = gas_standing_charge

In [ ]:
gas_tariff_cost_categories

In [ ]:
gas_unit_costs_each_year

In [ ]:
gas_standing_charges_each_year

In [ ]:
# creating time series of electricity costs
electricity_unit_costs_each_year = {}
electricity_standing_charges_each_year = {}
for year, electricity_tariff in electricity_tariff_each_year.items():

    # Cost component category mapping
    electricity_tariff_cost_components_category_map = {
        "nil": {
            electricity_tariff.df_nil: "wholesale",
            electricity_tariff.cm_nil: "wholesale",
            electricity_tariff.aa_nil: "other",
            electricity_tariff.pc_nil: "policy",
            electricity_tariff.nc_nil: "other",
            electricity_tariff.oc_nil: "other",
            electricity_tariff.smncc_nil: "other",
            electricity_tariff.ic_nil: "other",
            electricity_tariff.paac_nil: "other",
            electricity_tariff.pap_nil: "other",
            electricity_tariff.co_nil: "other",
            electricity_tariff.drc_nil: "other",
            electricity_tariff.ebit_nil: "other",
            electricity_tariff.hap_nil: "other",
            electricity_tariff.levelisation_nil: "other",
        },
        "variable": {
            electricity_tariff.df: "wholesale",
            electricity_tariff.cm: "wholesale",
            electricity_tariff.aa: "other",
            electricity_tariff.pc: "policy",
            electricity_tariff.nc: "other",
            electricity_tariff.oc: "other",
            electricity_tariff.smncc: "other",
            electricity_tariff.ic: "other",
            electricity_tariff.paac: "other",
            electricity_tariff.pap: "other",
            electricity_tariff.co: "other",
            electricity_tariff.drc: "other",
            electricity_tariff.ebit: "other",
            electricity_tariff.hap: "other",
            electricity_tariff.levelisation: "other",
        },
    }

    # Aggregate costs by category
    categories = ["wholesale", "policy", "other"]
    types = ["nil", "variable"]
    electricity_tariff_cost_categories = {
        f"{cat} cost ({type})" if cat == "policy" else f"{cat} ({type})": np.nansum(
            [
                component
                for component, category in electricity_tariff_cost_components_category_map.get(
                    type, {}
                ).items()
                if category == cat
            ]
        )
        for type in types
        for cat in categories
    }

    # look up wholesale price projection for year of interest
    # do not replace if year of interest is year of price cap period
    if year != electricity_tariff.price_cap_period.left.year:
        electricity_wholesale_projection = wholesale_prices_data.loc[
            (
                wholesale_prices_data["projection scenario"]
                == wholesale_price_projection_scenario
            )
            & (wholesale_prices_data["fuel"].str.contains("electricity", case=False)),
            year,
        ].iloc[
            0
        ]  # p/kWh

        # replace wholesale components with DESNZ projections
        electricity_tariff_cost_categories["wholesale (variable)"] = (
            electricity_wholesale_projection * 10
        )  # convert p/kWh to £/MWh
    else:
        pass

    # extract total unit costs and standing charges
    electricity_unit_cost = np.nansum(
        [
            value
            for category, value in electricity_tariff_cost_categories.items()
            if "(variable)" in category
        ]
    )
    electricity_standing_charge = np.nansum(
        [
            value
            for category, value in electricity_tariff_cost_categories.items()
            if "(nil)" in category
        ]
    )

    electricity_unit_costs_each_year[year] = electricity_unit_cost
    electricity_standing_charges_each_year[year] = electricity_standing_charge

In [ ]:
electricity_unit_costs_each_year, electricity_standing_charges_each_year

**Calculating total running costs**

In [ ]:
gas_boiler_running_costs_time_series = {}
for year in years_of_operation:
    gas_boiler_running_costs_time_series[year] = gas_unit_costs_each_year[year] * (
        boiler_gas_demand / 1_000
    )

In [ ]:
gas_boiler_running_costs_time_series

In [ ]:
# total lifetime running costs
sum(gas_boiler_running_costs_time_series.values()) + sum(
    gas_standing_charges_each_year.values()
)

In [ ]:
ashp_running_costs_time_series = {}
for year in years_of_operation:
    ashp_running_costs_time_series[year] = electricity_unit_costs_each_year[year] * (
        ashp_electricity_demand / 1_000
    )

In [ ]:
ashp_running_costs_time_series

In [ ]:
# total lifetime running costs
sum(ashp_running_costs_time_series.values())